# SPARC Example 15: Log-Slope Diagnostic

**EPS Research RAG Astrophysics Corpus — Unified HI Corpus v7.0**

The logarithmic slope d ln V / d ln R diagnoses the rotation curve shape:
- slope = +1.0: solid-body rotation (rising linearly)
- slope = 0.0:  flat rotation curve (constant V)
- slope = -0.5: Keplerian decline (falling as 1/sqrt(R))

Galaxies with outer slope near -0.5 are falling toward Keplerian —
these are the strongest candidates for the omega correction.

**Important note on corpus fidelity:** The `rotation_curve_corpus_v7_flat.csv` and `rotation_curve_corpus_v7.json` are **full-fidelity** — not a summary or veneer. The CSV contains every kinematic parameter published by Lelli et al. (2016) including per-galaxy inclination, distance uncertainties, mass-to-light ratios, and rotation curve statistics. The JSON adds full per-ring data: Vobs, Vgas, Vdisk, Vbul, errV at every radial point. This is the complete published dataset in a single machine-readable file.

**Corpus:** Flynn (2026), Zenodo DOI: 10.5281/zenodo.19563417  
**Source:** Lelli, McGaugh & Schombert (2016), AJ 152, 157  
**Dependencies:** Python 3, numpy, matplotlib, csv (standard library only)

In [ ]:
# ── Colab setup: download corpus from Zenodo ──────────────────
import os, urllib.request
CORPORA = {
    'rotation_curve_corpus_v7.json': 'https://zenodo.org/records/19563417/files/rotation_curve_corpus_v7.json',
}
for filename, url in CORPORA.items():
    if not os.path.exists(filename):
        print(f"Downloading {filename}...")
        urllib.request.urlretrieve(url, filename)
        print(f"  ✓ {filename}")
    else:
        print(f"  Already present: {filename}")
print("Ready.")


In [ ]:
import json
import numpy as np
import matplotlib.pyplot as plt

with open('rotation_curve_corpus_v7.json') as f:
    corpus = json.load(f)

# DDO161: compute slope at each ring
g = next(g for g in corpus['galaxies'] if g['galaxy'] == 'DDO161')
d = g['data']
R    = np.array([p['Rad']  for p in d])
Vobs = np.array([p['Vobs'] for p in d])

# Log slope via finite differences
lnR  = np.log(R)
lnV  = np.log(Vobs)
slope = np.gradient(lnV, lnR)

print("DDO161 log-slope profile:")
print(f"{'R (kpc)':>10}  {'Vobs':>8}  {'d ln V / d ln R':>16}")
for r, v, s in zip(R, Vobs, slope):
    print(f"{r:>10.2f}  {v:>8.2f}  {s:>16.3f}")
print(f"\nOuter slope: {slope[-1]:.3f}")
print(f"Keplerian slope: -0.5")

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(10, 4))
axes[0].plot(R, Vobs, 'o-', color='#1f77b4', linewidth=1.5)
axes[0].set_xlabel('Radius (kpc)', fontsize=11)
axes[0].set_ylabel(r'$V_{\rm obs}$ (km/s)', fontsize=11)
axes[0].set_title('DDO161 Rotation Curve', fontsize=10)

axes[1].plot(R, slope, 's-', color='#d62728', linewidth=1.5)
axes[1].axhline(0.0,  color='gray',  linestyle='--', linewidth=1, label='Flat (slope=0)')
axes[1].axhline(-0.5, color='orange',linestyle='--', linewidth=1, label='Keplerian (slope=-0.5)')
axes[1].axhline(1.0,  color='green', linestyle='--', linewidth=1, label='Solid-body (slope=+1)')
axes[1].set_xlabel('Radius (kpc)', fontsize=11)
axes[1].set_ylabel(r'd ln V / d ln R', fontsize=11)
axes[1].set_title('Log-Slope Profile', fontsize=10)
axes[1].legend(fontsize=8)

plt.suptitle('Log-Slope Diagnostic — DDO161\nEPS Research Corpus v7.0', fontsize=11)
plt.tight_layout()
plt.savefig('ex15_log_slope.png', dpi=150, bbox_inches='tight')
plt.show()